# Quantitative Momentum Strategy

"Momentum investing" means investing in the stocks that have increased in price the most.

For this project, we're going to build an investing strategy that selects the 50 stocks with the highest price momentum. From there, we will calculate recommended trades for an equal-weight portfolio of these 50 stocks.


## Library Imports

The first thing we need to do is import the open-source software libraries that we'll be using in this tutorial.

In [2]:
%pip install yfinance

Note: you may need to restart the kernel to use updated packages.


In [18]:
import numpy as np
import pandas as pd
import requests
import math
from scipy.stats import percentileofscore as score
import xlsxwriter
import yfinance as yf

## Importing Our List of Stocks

As before, we'll need to import our list of stocks and our API token before proceeding. Make sure the `.csv` file is still in your working directory and import it with the following command:

In [4]:
stocks = pd.read_csv('sp_500_stocks.csv')

## Making Our First API Call

It's now time to make the first version of our momentum screener!

We need to get one-year price returns for each stock in the universe. Here's how.

In [5]:
symbol = 'AAPL'
ticker = yf.Ticker(symbol)
data = ticker.info
data

{'address1': 'One Apple Park Way',
 'city': 'Cupertino',
 'state': 'CA',
 'zip': '95014',
 'country': 'United States',
 'phone': '(408) 996-1010',
 'website': 'https://www.apple.com',
 'industry': 'Consumer Electronics',
 'industryKey': 'consumer-electronics',
 'industryDisp': 'Consumer Electronics',
 'sector': 'Technology',
 'sectorKey': 'technology',
 'sectorDisp': 'Technology',
 'longBusinessSummary': 'Apple Inc. designs, manufactures, and markets smartphones, personal computers, tablets, wearables, and accessories worldwide. The company offers iPhone, a line of smartphones; Mac, a line of personal computers; iPad, a line of multi-purpose tablets; and wearables, home, and accessories comprising AirPods, Apple Vision Pro, Apple TV, Apple Watch, Beats products, and HomePod, as well as Apple branded and third-party accessories. It also provides AppleCare support and cloud services; and operates various platforms, including the App Store that allow customers to discover and download app

## Parsing Our API Call

This API call has all the information we need. We can parse it using the same square-bracket notation as in the first project of this course. Here is an example.

In [6]:
data['currentPrice']

260.645

## Executing A Batch API Call & Building Our DataFrame

Just like in our first project, it's now time to execute several batch API calls and add the information we need to our DataFrame.

We'll start by running the following code cell, which contains some code we already built last time that we can re-use for this project. More specifically, it contains a function called `chunks` that we can use to divide our list of securities into groups of 100.

In [7]:
# Function sourced from 
# https://stackoverflow.com/questions/312443/how-do-you-split-a-list-into-evenly-sized-chunks
def chunks(lst, n):
    """Yield successive n-sized chunks from lst."""
    for i in range(0, len(lst), n):
        yield lst[i:i + n]   
        
symbol_groups = list(chunks(stocks['Ticker'], 100))
symbol_strings = []
for i in range(0, len(symbol_groups)):
    symbol_strings.append(','.join(symbol_groups[i]))
#     print(symbol_strings[i])

my_columns = ['Ticker', 'Price', 'One-Year Price Return', 'Number of Shares to Buy']

Now we need to create a blank DataFrame and add our data to the data frame one-by-one.

In [8]:
final_dataframe = pd.DataFrame(columns = my_columns)
rows = []

for symbol_group in symbol_groups:
    tickers = yf.Tickers(' '.join(symbol_group))  # replaces batch_api_call_url
    
    for symbol in symbol_group:           # replaces symbol_string.split(',')
        try:
            data = tickers.tickers[symbol].info
            rows.append([
                symbol,
                data.get('currentPrice', 'N/A'),        # replaces data[symbol]['price']
                data.get('52WeekChange', 'N/A'),         # replaces data[symbol]['price']['year1ChangePercent']
                'N/A'
            ])
        except Exception as e:
            print(f"Skipping {symbol}: {e}")

final_dataframe = pd.DataFrame(rows, columns=my_columns)
final_dataframe

HTTP Error 404: Not Found{"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: ANSS"}}}
HTTP Error 404: Not Found{"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: CMA"}}}
HTTP Error 404: Not Found{"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: COG"}}}
HTTP Error 404: Not Found{"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: FRC"}}}
HTTP Error 404: Not Found{"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: HBI"}}}
HTTP Error 404: Not Found{"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: HES"}}}
HTTP Error 404: Not Found{"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: IPG"}}}
HTTP Error 404: Not Found{"quoteSummary":{"result":nul

,Ticker,Price,One-Year Price Return,Number of Shares to Buy
0,A,121.5,0.172249,N/A
1,AAL,12.175,0.287686,N/A
2,AAP,55.775,0.813795,N/A
3,AAPL,260.75,0.332321,N/A
4,ABBV,206.995,0.22472,N/A
...,...,...,...,...
500,YUM,161.22,0.133268,N/A
501,ZBH,95.235,-0.004333,N/A
502,ZBRA,228.59,0.008631,N/A
503,ZION,61.67,0.42274,N/A


## Removing Low-Momentum Stocks

The investment strategy that we're building seeks to identify the 50 highest-momentum stocks in the S&P 500.

Because of this, the next thing we need to do is remove all the stocks in our DataFrame that fall below this momentum threshold. We'll sort the DataFrame by the stocks' one-year price return, and drop all stocks outside the top 50.


In [9]:
final_dataframe['One-Year Price Return'] = pd.to_numeric(final_dataframe['One-Year Price Return'], errors='coerce')

final_dataframe.sort_values('One-Year Price Return', ascending=False, inplace=True)
final_dataframe = final_dataframe[:50]
final_dataframe.reset_index(inplace = True)
final_dataframe

,index,Ticker,Price,One-Year Price Return,Number of Shares to Buy
0,479,WDC,360.28,9.281302,N/A
1,422,STX,510.53,6.325460,N/A
2,324,MU,450.64,5.716573,N/A
3,202,GLW,168.86,3.186531,N/A
4,288,LRCX,264.735,3.181916,N/A
5,23,ALB,188.4,2.611566,N/A
6,242,INTC,65.335,2.318253,N/A
7,31,AMD,257.29,1.889002,N/A
8,194,FTI,71.56,1.881152,N/A
9,29,AMAT,387.94,1.863015,N/A


## Calculating the Number of Shares to Buy

Just like in the last project, we now need to calculate the number of shares we need to buy. The one change we're going to make is wrapping this functionality inside a function, since we'll be using it again later in this Jupyter Notebook.

Since we've already done most of the work on this, try to complete the following two code cells without watching me do it first!

In [10]:
def portfolio_input():
    global portfolio_size
    portfolio_size = input('Enter the size of your portfolio:')

    try:
        float(portfolio_size)
    except ValueError:
        print('That is not a number! \nPlease try again:')
        portfolio_size = input('Enter the size of your portfolio: ')

portfolio_input()
print(portfolio_size)

10000000


In [11]:
final_dataframe['Number of Shares to Buy'] = 0.0

position_size = float(portfolio_size) / len(final_dataframe.index)
for i in range(0, len(final_dataframe)):
    final_dataframe.loc[i, 'Number of Shares to Buy'] = math.floor(position_size / final_dataframe.loc[i, 'Price'])

final_dataframe

,index,Ticker,Price,One-Year Price Return,Number of Shares to Buy
0,479,WDC,360.28,9.281302,555.0
1,422,STX,510.53,6.325460,391.0
2,324,MU,450.64,5.716573,443.0
3,202,GLW,168.86,3.186531,1184.0
4,288,LRCX,264.735,3.181916,755.0
5,23,ALB,188.4,2.611566,1061.0
6,242,INTC,65.335,2.318253,3061.0
7,31,AMD,257.29,1.889002,777.0
8,194,FTI,71.56,1.881152,2794.0
9,29,AMAT,387.94,1.863015,515.0


## Building a Better (and More Realistic) Momentum Strategy

Real-world quantitative investment firms differentiate between "high quality" and "low quality" momentum stocks:

* High-quality momentum stocks show "slow and steady" outperformance over long periods of time
* Low-quality momentum stocks might not show any momentum for a long time, and then surge upwards.

The reason why high-quality momentum stocks are preferred is because low-quality momentum can often be cause by short-term news that is unlikely to be repeated in the future (such as an FDA approval for a biotechnology company).

To identify high-quality momentum, we're going to build a strategy that selects stocks from the highest percentiles of: 

* 1-month price returns
* 3-month price returns
* 6-month price returns
* 1-year price returns

Let's start by building our DataFrame. You'll notice that I use the abbreviation `hqm` often. It stands for `high-quality momentum`.

In [15]:
stocks = pd.read_csv('sp_500_stocks.csv')

hqm_columns = [
    'Ticker', 
    'Price', 
    'Number of Shares to Buy', 
    'One-Year Price Return', 
    'One-Year Return Percentile',
    'Six-Month Price Return',
    'Six-Month Return Percentile',
    'Three-Month Price Return',
    'Three-Month Return Percentile',
    'One-Month Price Return',
    'One-Month Return Percentile',
    'HQM Score'
]

# Download all historical data in one call
all_history = yf.download(
    stocks['Ticker'].tolist(),
    period='2y',
    group_by='ticker',
    auto_adjust=True,
    threads=True
)

# Helper function to calculate returns
def get_return(hist, days):
    if len(hist) >= days:
        return (hist.iloc[-1] - hist.iloc[-days]) / hist.iloc[-days]
    return None

# Build hqm dataframe
rows = []

for symbol in stocks['Ticker']:
    try:
        ticker = yf.Ticker(symbol)
        data = ticker.info
        
        hist = all_history[symbol]['Close'].dropna()
        
        rows.append([
            symbol,
            data.get('currentPrice', None),
            None,
            get_return(hist, 252), None,  # One-Year
            get_return(hist, 126), None,  # Six-Month
            get_return(hist, 63),  None,  # Three-Month
            get_return(hist, 21),  None,  # One-Month
            None
        ])
        print(f"Fetched: {symbol}")
    except Exception as e:
        print(f"Skipping {symbol}: {e}")

hqm_dataframe = pd.DataFrame(rows, columns=hqm_columns)
hqm_dataframe

[*****                 11%                       ]  55 of 505 completed$DFS: possibly delisted; no price data found  (period=2y) (Yahoo error = "No data found, symbol may be delisted")
[******                12%                       ]  63 of 505 completed$DISCA: possibly delisted; no price data found  (period=2y) (Yahoo error = "No data found, symbol may be delisted")
[*******               14%                       ]  73 of 505 completed$ANSS: possibly delisted; no price data found  (period=2y) (Yahoo error = "No data found, symbol may be delisted")
[********              17%                       ]  84 of 505 completed$HES: possibly delisted; no price data found  (period=2y) (Yahoo error = "No data found, symbol may be delisted")
[*********             19%                       ]  96 of 505 completed$DISCK: possibly delisted; no price data found  (period=2y) (Yahoo error = "No data found, symbol may be delisted")
[*********             19%                       ]  98 of 505 complete

Fetched: A
Fetched: AAL
Fetched: AAP
Fetched: AAPL
Fetched: ABBV
Fetched: ABC
Fetched: ABMD
Fetched: ABT
Fetched: ACN
Fetched: ADBE
Fetched: ADI
Fetched: ADM
Fetched: ADP
Fetched: ADSK
Fetched: AEE
Fetched: AEP
Fetched: AES
Fetched: AFL
Fetched: AIG
Fetched: AIV
Fetched: AIZ
Fetched: AJG
Fetched: AKAM
Fetched: ALB
Fetched: ALGN
Fetched: ALK
Fetched: ALL
Fetched: ALLE
Fetched: ALXN
Fetched: AMAT
Fetched: AMCR
Fetched: AMD
Fetched: AME
Fetched: AMGN
Fetched: AMP
Fetched: AMT
Fetched: AMZN
Fetched: ANET


HTTP Error 404: Not Found{"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: ANSS"}}}


Fetched: ANSS
Fetched: ANTM
Fetched: AON
Fetched: AOS
Fetched: APA
Fetched: APD
Fetched: APH
Fetched: APTV
Fetched: ARE
Fetched: ATO
Fetched: ATVI
Fetched: AVB
Fetched: AVGO
Fetched: AVY
Fetched: AWK
Fetched: AXP
Fetched: AZO
Fetched: BA
Fetched: BAC
Fetched: BAX
Fetched: BBY
Fetched: BDX
Fetched: BEN
Fetched: BF.B
Fetched: BIIB
Fetched: BIO
Fetched: BK
Fetched: BKNG
Fetched: BKR
Fetched: BLK
Fetched: BLL
Fetched: BMY
Fetched: BR
Fetched: BRK.B
Fetched: BSX
Fetched: BWA
Fetched: BXP
Fetched: C
Fetched: CAG
Fetched: CAH
Fetched: CARR
Fetched: CAT
Fetched: CB
Fetched: CBOE
Fetched: CBRE
Fetched: CCI
Fetched: CCL
Fetched: CDNS
Fetched: CDW
Fetched: CE
Fetched: CERN
Fetched: CF
Fetched: CFG
Fetched: CHD
Fetched: CHRW
Fetched: CHTR
Fetched: CI
Fetched: CINF
Fetched: CL
Fetched: CLX


HTTP Error 404: Not Found{"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: CMA"}}}


...

HTTP Error 404: Not Found{"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: TWTR"}}}


Fetched: TWTR
Fetched: TXN
Fetched: TXT
Fetched: TYL
Fetched: UA
Fetched: UAA
Fetched: UAL
Fetched: UDR
Fetched: UHS
Fetched: ULTA
Fetched: UNH
Fetched: UNM
Fetched: UNP
Fetched: UPS
Fetched: URI
Fetched: USB
Fetched: V
Fetched: VAR
Fetched: VFC
Fetched: VIAC
Fetched: VLO
Fetched: VMC
Fetched: VNO
Fetched: VRSK
Fetched: VRSN
Fetched: VRTX
Fetched: VTR
Fetched: VZ
Fetched: WAB
Fetched: WAT


HTTP Error 404: Not Found{"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: WBA"}}}


Fetched: WBA
Fetched: WDC
Fetched: WEC
Fetched: WELL
Fetched: WFC
Fetched: WHR
Fetched: WLTW
Fetched: WM
Fetched: WMB
Fetched: WMT
Fetched: WRB
Fetched: WRK
Fetched: WST
Fetched: WU
Fetched: WY
Fetched: WYNN
Fetched: XEL
Fetched: XLNX
Fetched: XOM
Fetched: XRAY
Fetched: XRX
Fetched: XYL
Fetched: YUM
Fetched: ZBH
Fetched: ZBRA
Fetched: ZION
Fetched: ZTS


,Ticker,Price,Number of Shares to Buy,One-Year Price Return,One-Year Return Percentile,Six-Month Price Return,Six-Month Return Percentile,Three-Month Price Return,Three-Month Return Percentile,One-Month Price Return,One-Month Return Percentile,HQM Score
0,A,121.4250,None,0.163863,None,-0.116744,None,-0.165933,None,0.079442,None,None
1,AAL,12.1201,None,0.265658,None,-0.005332,None,-0.199141,None,0.116483,None,None
2,AAP,56.1600,None,0.741720,None,0.033216,None,0.315519,None,0.086469,None,None
3,AAPL,263.0039,None,0.304049,None,0.063289,None,0.012448,None,0.034300,None,None
4,ABBV,206.1650,None,0.191463,None,-0.085120,None,-0.062030,None,-0.060498,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...
500,YUM,161.0500,None,0.124307,None,0.129035,None,0.029671,None,-0.016128,None,None
501,ZBH,95.5300,None,-0.053955,None,-0.024258,None,0.072717,None,0.034687,None,None
502,ZBRA,229.1350,None,-0.011520,None,-0.207239,None,-0.127271,None,0.095858,None,None
503,ZION,61.7600,None,0.488022,None,0.135398,None,0.058259,None,0.143465,None,None


## Calculating Momentum Percentiles

We now need to calculate momentum percentile scores for every stock in the universe. More specifically, we need to calculate percentile scores for the following metrics for every stock:

* `One-Year Price Return`
* `Six-Month Price Return`
* `Three-Month Price Return`
* `One-Month Price Return`

Here's how we'll do this:

In [35]:
from scipy.stats import percentileofscore as score

time_periods = [
    'One-Year',
    'Six-Month',
    'Three-Month',
    'One-Month'
]

# Convert return columns to numeric first
for time_period in time_periods:
    hqm_dataframe[f'{time_period} Price Return'] = pd.to_numeric(
        hqm_dataframe[f'{time_period} Price Return'], errors='coerce'
    )

for row in hqm_dataframe.index:
    for time_period in time_periods:
        hqm_dataframe.loc[row, f'{time_period} Return Percentile'] = score(
            hqm_dataframe[f'{time_period} Price Return'].dropna(),
            hqm_dataframe.loc[row, f'{time_period} Price Return']
        ) / 100

# Print each percentile score to make sure it was calculated properly
for time_period in time_periods:
    print(hqm_dataframe[f'{time_period} Return Percentile'])

hqm_dataframe

0     0.98
1     0.94
2      1.0
3     0.88
4     0.92
5     0.72
6     0.78
7     0.86
8     0.82
9     0.56
10    0.62
11     0.8
12     0.7
13    0.52
14    0.84
15     0.9
16     0.2
17    0.28
18    0.32
19    0.68
20    0.76
21    0.46
22    0.26
23     0.1
24    0.22
25     0.5
26    0.58
27    0.02
28    0.24
29    0.64
30    0.44
31    0.16
32    0.18
33     0.6
34    0.54
35    0.14
36    0.74
37    0.42
38     0.4
39    0.36
40    0.06
41    0.38
42    0.12
43    0.04
44    0.34
45    0.48
46    0.08
47    0.96
48    0.66
49     0.3
Name: One-Year Return Percentile, dtype: object
0     0.98
1     0.92
2      1.0
3     0.82
4     0.86
5      0.9
6     0.76
7     0.94
8      0.8
9     0.74
10    0.34
11    0.52
12    0.56
13    0.54
14    0.06
15    0.88
16    0.68
17    0.64
18    0.44
19    0.04
20     0.7
21    0.78
22    0.66
23    0.84
24    0.36
25     0.2
26    0.02
27    0.58
28    0.42
29     0.4
30    0.16
31    0.48
32    0.24
33    0.46
34     0.3
35    0.32
36    

,level_0,index,Ticker,Price,Number of Shares to Buy,One-Year Price Return,One-Year Return Percentile,Six-Month Price Return,Six-Month Return Percentile,Three-Month Price Return,Three-Month Return Percentile,One-Month Price Return,One-Month Return Percentile,HQM Score
0,0,422,STX,513.3500,0,6.163813,0.98,1.439244,0.98,0.648734,0.96,0.222700,0.92,0.994456
1,1,202,GLW,168.3016,0,3.109083,0.94,1.001301,0.92,0.870684,1.0,0.296790,0.96,0.994453
2,2,479,WDC,360.0900,0,9.080735,1.0,2.182192,1.0,0.673080,0.98,0.145725,0.72,0.987251
3,3,242,INTC,65.3300,1,2.216741,0.88,0.833623,0.82,0.340969,0.76,0.482796,1.0,0.983917
4,4,288,LRCX,264.0300,0,2.921391,0.92,0.915421,0.86,0.266759,0.7,0.166468,0.78,0.975051
5,5,263,KEYS,325.4900,0,1.386465,0.72,0.974581,0.9,0.554394,0.92,0.132612,0.64,0.968941
6,6,266,KLAC,1725.3900,0,1.596550,0.78,0.688259,0.76,0.205133,0.56,0.165504,0.76,0.960075
7,7,194,FTI,72.0100,1,1.849814,0.86,1.002802,0.94,0.378187,0.84,0.101010,0.44,0.953428
8,8,29,AMAT,387.4200,0,1.704530,0.82,0.781308,0.8,0.284916,0.72,0.099189,0.42,0.942894
9,9,179,FCX,68.7900,1,1.068530,0.56,0.650564,0.74,0.143771,0.34,0.185316,0.86,0.938438


## Calculating the HQM Score

We'll now calculate our `HQM Score`, which is the high-quality momentum score that we'll use to filter for stocks in this investing strategy.

The `HQM Score` will be the arithmetic mean of the 4 momentum percentile scores that we calculated in the last section.

To calculate arithmetic mean, we will use the `mean` function from Python's built-in `statistics` module.

In [36]:
from statistics import mean

for row in hqm_dataframe.index:
    momentum_percentiles = []
    for time_period in time_periods:
        momentum_percentiles.append(hqm_dataframe.loc[row, f'{time_period} Return Percentile'])
    hqm_dataframe.loc[row, 'HQM Score'] = mean(momentum_percentiles)

hqm_dataframe

,level_0,index,Ticker,Price,Number of Shares to Buy,One-Year Price Return,One-Year Return Percentile,Six-Month Price Return,Six-Month Return Percentile,Three-Month Price Return,Three-Month Return Percentile,One-Month Price Return,One-Month Return Percentile,HQM Score
0,0,422,STX,513.3500,0,6.163813,0.98,1.439244,0.98,0.648734,0.96,0.222700,0.92,0.96
1,1,202,GLW,168.3016,0,3.109083,0.94,1.001301,0.92,0.870684,1.0,0.296790,0.96,0.955
2,2,479,WDC,360.0900,0,9.080735,1.0,2.182192,1.0,0.673080,0.98,0.145725,0.72,0.925
3,3,242,INTC,65.3300,1,2.216741,0.88,0.833623,0.82,0.340969,0.76,0.482796,1.0,0.865
4,4,288,LRCX,264.0300,0,2.921391,0.92,0.915421,0.86,0.266759,0.7,0.166468,0.78,0.815
5,5,263,KEYS,325.4900,0,1.386465,0.72,0.974581,0.9,0.554394,0.92,0.132612,0.64,0.795
6,6,266,KLAC,1725.3900,0,1.596550,0.78,0.688259,0.76,0.205133,0.56,0.165504,0.76,0.715
7,7,194,FTI,72.0100,1,1.849814,0.86,1.002802,0.94,0.378187,0.84,0.101010,0.44,0.77
8,8,29,AMAT,387.4200,0,1.704530,0.82,0.781308,0.8,0.284916,0.72,0.099189,0.42,0.69
9,9,179,FCX,68.7900,1,1.068530,0.56,0.650564,0.74,0.143771,0.34,0.185316,0.86,0.625


## Selecting the 50 Best Momentum Stocks

As before, we can identify the 50 best momentum stocks in our universe by sorting the DataFrame on the `HQM Score` column and dropping all but the top 50 entries.

In [37]:
hqm_dataframe.sort_values('HQM Score', ascending = False, inplace = True)
hqm_dataframe = hqm_dataframe[:50]
hqm_dataframe.reset_index(drop = True, inplace = True)
hqm_dataframe

,level_0,index,Ticker,Price,Number of Shares to Buy,One-Year Price Return,One-Year Return Percentile,Six-Month Price Return,Six-Month Return Percentile,Three-Month Price Return,Three-Month Return Percentile,One-Month Price Return,One-Month Return Percentile,HQM Score
0,0,422,STX,513.3500,0,6.163813,0.98,1.439244,0.98,0.648734,0.96,0.222700,0.92,0.96
1,1,202,GLW,168.3016,0,3.109083,0.94,1.001301,0.92,0.870684,1.0,0.296790,0.96,0.955
2,2,479,WDC,360.0900,0,9.080735,1.0,2.182192,1.0,0.673080,0.98,0.145725,0.72,0.925
3,3,242,INTC,65.3300,1,2.216741,0.88,0.833623,0.82,0.340969,0.76,0.482796,1.0,0.865
4,4,288,LRCX,264.0300,0,2.921391,0.92,0.915421,0.86,0.266759,0.7,0.166468,0.78,0.815
5,5,263,KEYS,325.4900,0,1.386465,0.72,0.974581,0.9,0.554394,0.92,0.132612,0.64,0.795
6,7,194,FTI,72.0100,1,1.849814,0.86,1.002802,0.94,0.378187,0.84,0.101010,0.44,0.77
7,6,266,KLAC,1725.3900,0,1.596550,0.78,0.688259,0.76,0.205133,0.56,0.165504,0.76,0.715
8,8,29,AMAT,387.4200,0,1.704530,0.82,0.781308,0.8,0.284916,0.72,0.099189,0.42,0.69
9,47,324,MU,452.4700,0,5.381897,0.96,1.419185,0.96,0.356982,0.8,-0.020230,0.02,0.685


## Calculating the Number of Shares to Buy

We'll use the `portfolio_input` function that we created earlier to accept our portfolio size. Then we will use similar logic in a `for` loop to calculate the number of shares to buy for each stock in our investment universe.

In [40]:
portfolio_input()

In [44]:
hqm_dataframe['Number of Shares to Buy'] = 0.0

portfolio_size = float(input('Enter the size of your portfolio: '))
position_size = portfolio_size / len(hqm_dataframe.index)

for i in range(0, len(hqm_dataframe)):
    hqm_dataframe.loc[i, 'Number of Shares to Buy'] = math.floor(
        position_size / hqm_dataframe.loc[i, 'Price']
    )

hqm_dataframe

,level_0,index,Ticker,Price,Number of Shares to Buy,One-Year Price Return,One-Year Return Percentile,Six-Month Price Return,Six-Month Return Percentile,Three-Month Price Return,Three-Month Return Percentile,One-Month Price Return,One-Month Return Percentile,HQM Score
0,0,422,STX,513.3500,0.0,6.163813,0.98,1.439244,0.98,0.648734,0.96,0.222700,0.92,0.96
1,1,202,GLW,168.3016,1.0,3.109083,0.94,1.001301,0.92,0.870684,1.0,0.296790,0.96,0.955
2,2,479,WDC,360.0900,0.0,9.080735,1.0,2.182192,1.0,0.673080,0.98,0.145725,0.72,0.925
3,3,242,INTC,65.3300,3.0,2.216741,0.88,0.833623,0.82,0.340969,0.76,0.482796,1.0,0.865
4,4,288,LRCX,264.0300,0.0,2.921391,0.92,0.915421,0.86,0.266759,0.7,0.166468,0.78,0.815
5,5,263,KEYS,325.4900,0.0,1.386465,0.72,0.974581,0.9,0.554394,0.92,0.132612,0.64,0.795
6,7,194,FTI,72.0100,2.0,1.849814,0.86,1.002802,0.94,0.378187,0.84,0.101010,0.44,0.77
7,6,266,KLAC,1725.3900,0.0,1.596550,0.78,0.688259,0.76,0.205133,0.56,0.165504,0.76,0.715
8,8,29,AMAT,387.4200,0.0,1.704530,0.82,0.781308,0.8,0.284916,0.72,0.099189,0.42,0.69
9,47,324,MU,452.4700,0.0,5.381897,0.96,1.419185,0.96,0.356982,0.8,-0.020230,0.02,0.685


## Formatting Our Excel Output

We will be using the XlsxWriter library for Python to create nicely-formatted Excel files.

XlsxWriter is an excellent package and offers tons of customization. However, the tradeoff for this is that the library can seem very complicated to new users. Accordingly, this section will be fairly long because I want to do a good job of explaining how XlsxWriter works.

In [52]:
writer = pd.ExcelWriter('momentum_strategy.xlsx', engine='xlsxwriter')
hqm_dataframe.to_excel(writer, sheet_name = "Momentum Strategy", index = False)

## Creating the Formats We'll Need For Our .xlsx File

You'll recall from our first project that formats include colors, fonts, and also symbols like % and $. We'll need four main formats for our Excel document:

* String format for tickers
* \$XX.XX format for stock prices
* \$XX,XXX format for market capitalization
* Integer format for the number of shares to purchase

Since we already built our formats in the last section of this course, I've included them below for you. Run this code cell before proceeding.

In [53]:
background_color = '#0a0a23'
font_color = '#ffffff'

string_template = writer.book.add_format(
        {
            'font_color': font_color,
            'bg_color': background_color,
            'border': 1
        }
    )

dollar_template = writer.book.add_format(
        {
            'num_format':'$0.00',
            'font_color': font_color,
            'bg_color': background_color,
            'border': 1
        }
    )

integer_template = writer.book.add_format(
        {
            'num_format':'0',
            'font_color': font_color,
            'bg_color': background_color,
            'border': 1
        }
    )

percent_template = writer.book.add_format(
        {
            'num_format':'0.0%',
            'font_color': font_color,
            'bg_color': background_color,
            'border': 1
        }
   )

In [67]:
import os

output_path = os.path.expanduser('~/Downloads/momentum_strategy.xlsx')

writer = pd.ExcelWriter(output_path, engine='xlsxwriter')
hqm_dataframe.to_excel(writer, sheet_name='Momentum Strategy', index=False)

column_formats = {
    'A': ['Ticker', string_template], 
    'B': ['Price', dollar_template], 
    'C': ['Number of Shares to Buy', integer_template], 
    'D': ['One-Year Price Return', percent_template], 
    'E': ['One-Year Return Percentile', percent_template],
    'F': ['Six-Month Price Return', percent_template],
    'G': ['Six-Month Return Percentile', percent_template],
    'H': ['Three-Month Price Return', percent_template],
    'I': ['Three-Month Return Percentile', percent_template],
    'J': ['One-Month Price Return', percent_template],
    'K': ['One-Month Return Percentile', percent_template],
    'L': ['HQM Score', percent_template]
}

for column in column_formats.keys():
    writer.sheets['Momentum Strategy'].set_column(f'{column}:{column}', 20, column_formats[column][1])
    writer.sheets['Momentum Strategy'].write(f'{column}1', column_formats[column][0], string_template)

## Saving Our Excel Output

As before, saving our Excel output is very easy:

In [68]:
writer.close()
print(f"File saved to: {output_path}")

File saved to: /Users/favourstjohn/Downloads/momentum_strategy.xlsx
